# 第 18 章 RBC モデル: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/18-rbc-model.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
alpha = 0.36        # 資本分配率 α
beta = 0.99         # 割引因子 β
delta = 0.025       # 減耗率 δ
rho = 0.551         # TFP のショックの持続性 ρ
shock = 0.01        # 1 期間だけの TFP ショック（0.01 は 1%）
endogenous_labour = False  # True にすると、労働時間も、内生的に決まる

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 定常状態と、ショックへの反応

In [ ]:
from lib.chapter18.rbc import (
    RbcParameters,
    impulse_response,
    solve_rbc,
    steady_state,
)

params = RbcParameters(alpha=alpha, beta=beta, delta=delta, rho=rho)
ss = steady_state(params)
print(f"実質利子率 R: {ss['r']:.4f}、K ÷ Y: {ss['k'] / ss['y']:.2f}")

df = impulse_response(solve_rbc(params, endogenous_labour), shock=shock)
first = df.iloc[3]
for name, label in (("y", "産出量"), ("c", "消費"), ("i", "投資")):
    print(f"ショックの期の{label}: {first[name]:.2f} %")

## 図: ショックへの反応

In [ ]:
from lib.chapter18.plot import plot_rbc_responses

plot_rbc_responses(df)

## 試してみよう

- `endogenous_labour = True` にすると、
  ショックの期の産出量は 1.00 % から 1.65 % に、投資は 3.65 % から 6.05 % に、
  大きくなります
- `rho = 0.9` にすると、消費は 0.09 % から 0.21 % に、
  投資は 3.28 % に変わります（ショックが長く続くと、消費の反応は大きくなります）
- `alpha = 1 / 3` にすると、K ÷ Y は 9.50 です
- `shock = 0.02` にすると、産出量 2.00 %、消費 0.17 %、投資 7.29 % で、
  反応はほぼ 2 倍です